# 📊 01 — Dataset Exploration
**Tujuan:** Eksplorasi dataset TACO sebelum training.

Notebook ini mencakup:
1. Download dan verifikasi dataset
2. Analisis distribusi kelas
3. Analisis ukuran objek (untuk menentukan threshold 'objek kecil')
4. Visualisasi contoh anotasi
5. Pembuatan struktur YOLO dataset


In [ ]:
import sys
sys.path.insert(0, '..')  # Add project root to path

import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from collections import Counter
import cv2
from PIL import Image

# Project config
import yaml
with open('../config/config.yaml') as f:
    cfg = yaml.safe_load(f)

print('✅ Config loaded')
print(f'Project: {cfg["project"]["name"]}')
print(f'Classes: {cfg["dataset"]["class_names"]}')


## Step 1: Download TACO Dataset

In [ ]:
import subprocess
import os

os.makedirs('../logs', exist_ok=True)

# Download annotations only first (gambar memerlukan waktu lebih lama)
result = subprocess.run(
    ['python', '../data/download_taco.py', '--skip-images'],
    cwd='..', capture_output=True, text=True
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])


## Step 2: Load and Analyze Annotations

In [ ]:
ann_path = Path('../data/raw/annotations.json')

if not ann_path.exists():
    print('⚠️  Annotations not found. Run download first.')
else:
    with open(ann_path) as f:
        taco_data = json.load(f)
    
    print(f'📊 TACO Dataset Statistics:')
    print(f'   Images     : {len(taco_data["images"])}')
    print(f'   Annotations: {len(taco_data["annotations"])}')
    print(f'   Categories : {len(taco_data["categories"])}')
    print()
    print('Categories:')
    for cat in taco_data['categories'][:15]:
        print(f'   [{cat["id"]:3d}] {cat["name"]}')


## Step 3: Class Distribution Analysis

In [ ]:
# Count annotations per original TACO category
cat_id_to_name = {c['id']: c['name'] for c in taco_data['categories']}
class_counts = Counter(
    cat_id_to_name[ann['category_id']] 
    for ann in taco_data['annotations']
    if ann['category_id'] in cat_id_to_name
)

# Plot top 20
top_20 = class_counts.most_common(20)
names, counts = zip(*top_20)

fig, ax = plt.subplots(figsize=(14, 6))
colors = plt.cm.tab20(np.linspace(0, 1, 20))
bars = ax.barh(names[::-1], counts[::-1], color=colors)
ax.set_title('TACO Dataset — Top 20 Classes', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Annotations')
for bar, count in zip(bars, counts[::-1]):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            str(count), va='center')
plt.tight_layout()
plt.savefig('../results/visualizations/class_distribution_taco.png', dpi=150)
plt.show()


## Step 4: Object Size Analysis

In [ ]:
# Analyze bounding box areas to understand small object prevalence
img_id_to_hw = {
    img['id']: (img.get('height', 480), img.get('width', 640))
    for img in taco_data['images']
}

areas = []
relative_areas = []
for ann in taco_data['annotations']:
    if 'bbox' not in ann or len(ann['bbox']) != 4:
        continue
    _, _, w, h = ann['bbox']
    area = w * h
    areas.append(area)
    
    img_h, img_w = img_id_to_hw.get(ann['image_id'], (480, 640))
    img_area = img_h * img_w
    if img_area > 0:
        relative_areas.append(area / img_area * 100)

areas = np.array(areas)
small_mask = areas < 1024       # < 32×32
medium_mask = (areas >= 1024) & (areas < 9216)  # 32×32 — 96×96
large_mask = areas >= 9216

print('📐 Object Size Distribution:')
print(f'   Total annotations: {len(areas)}')
print(f'   Small  (< 32×32)  : {small_mask.sum()} ({small_mask.mean()*100:.1f}%)')
print(f'   Medium (32×96 px) : {medium_mask.sum()} ({medium_mask.mean()*100:.1f}%)')
print(f'   Large  (> 96×96)  : {large_mask.sum()} ({large_mask.mean()*100:.1f}%)')
print(f'   Median area       : {np.median(areas):.0f} px²')
print(f'   Mean area         : {np.mean(areas):.0f} px²')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of absolute areas
axes[0].hist(np.clip(areas, 0, 50000), bins=60, color='steelblue', edgecolor='white')
axes[0].axvline(1024, color='red', linestyle='--', label='Small threshold (32²)')
axes[0].axvline(9216, color='orange', linestyle='--', label='Medium threshold (96²)')
axes[0].set_title('BBox Area Distribution')
axes[0].set_xlabel('Area (pixels²)')
axes[0].legend()

# Pie chart
sizes = [small_mask.sum(), medium_mask.sum(), large_mask.sum()]
labels = [f'Small\n({small_mask.mean()*100:.1f}%)', 
          f'Medium\n({medium_mask.mean()*100:.1f}%)',
          f'Large\n({large_mask.mean()*100:.1f}%)']
axes[1].pie(sizes, labels=labels, colors=['#e74c3c', '#f39c12', '#27ae60'],
            autopct='%1.1f%%', startangle=90)
axes[1].set_title('Object Size Categories')

plt.tight_layout()
Path('../results/visualizations').mkdir(parents=True, exist_ok=True)
plt.savefig('../results/visualizations/object_size_distribution.png', dpi=150)
plt.show()


## Step 5: Preprocess Dataset (COCO → YOLO)

In [ ]:
# Run preprocessing script
result = subprocess.run(
    ['python', '../data/preprocess.py',
     '--annotations', '../data/raw/annotations.json',
     '--images-dir', '../data/raw/images',
     '--output-dir', '../data/yolo'],
    cwd='..', capture_output=True, text=True
)
print(result.stdout[-5000:] if len(result.stdout) > 5000 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-2000:])


## Step 6: Verify YOLO Dataset Structure

In [ ]:
yolo_dir = Path('../data/yolo')

for split in ['train', 'val', 'test']:
    n_imgs = len(list((yolo_dir / 'images' / split).glob('*.jpg'))) if (yolo_dir / 'images' / split).exists() else 0
    n_lbls = len(list((yolo_dir / 'labels' / split).glob('*.txt'))) if (yolo_dir / 'labels' / split).exists() else 0
    print(f'{split:6s}: {n_imgs} images, {n_lbls} labels')

# Read data.yaml
data_yaml_path = yolo_dir / 'data.yaml'
if data_yaml_path.exists():
    with open(data_yaml_path) as f:
        data_cfg = yaml.safe_load(f)
    print(f'\ndata.yaml:')
    print(f'  Classes: {data_cfg["nc"]}')
    print(f'  Names  : {data_cfg["names"]}')


## Step 7: Sample Visualization

In [ ]:
# Visualize a few training samples with annotations
from src.utils.visualization import draw_ground_truth

train_img_dir = yolo_dir / 'images' / 'train'
train_lbl_dir = yolo_dir / 'labels' / 'train'

sample_images = list(train_img_dir.glob('*.jpg'))[:6]

if sample_images:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    class_names = data_cfg['names'] if data_yaml_path.exists() else [f'cls_{i}' for i in range(7)]
    
    for ax, img_path in zip(axes.flatten(), sample_images):
        img = cv2.imread(str(img_path))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        H, W = img.shape[:2]
        
        lbl_path = train_lbl_dir / (img_path.stem + '.txt')
        boxes = []
        cls_ids = []
        if lbl_path.exists():
            for line in lbl_path.read_text().strip().split('\n'):
                parts = line.split()
                if len(parts) == 5:
                    c, xc, yc, w, h = int(parts[0]), *map(float, parts[1:])
                    x1 = (xc - w/2) * W
                    y1 = (yc - h/2) * H
                    x2 = (xc + w/2) * W
                    y2 = (yc + h/2) * H
                    boxes.append([x1, y1, x2, y2])
                    cls_ids.append(c)
        
        if boxes:
            boxes = np.array(boxes, dtype=np.float32)
            cls_ids = np.array(cls_ids, dtype=np.int32)
            from src.utils.visualization import draw_detections
            vis = draw_detections(img, boxes, np.ones(len(boxes)), cls_ids,
                                  class_names=class_names, show_confidence=False)
            img_rgb = cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)
        
        ax.imshow(img_rgb)
        ax.set_title(img_path.stem[:20], fontsize=9)
        ax.axis('off')
    
    plt.suptitle('Sample Training Images with Annotations', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../results/visualizations/sample_annotations.png', dpi=150)
    plt.show()
else:
    print('No training images found. Run preprocessing first.')


---

✅ **Dataset exploration complete!**

Next step: Run `02_training.ipynb` to fine-tune YOLOv11n on the preprocessed dataset.